# Task 3 — SmallCNN Baseline Training

This notebook reproduces the fixed SmallCNN baseline in a managed Colab GPU runtime. New experiments live in a separate notebook so Run All here has one clear meaning: reproduce the original baseline.

Run the cells from top to bottom. The gender and usage training calls are separate foreground cells. Each call trains all five official folds, including fold 0, and writes every run plus its artifacts to Google Drive. Do not add `&`, `nohup`, or another background launcher.

Before starting, select a GPU runtime in Colab or the VS Code Colab extension.


## 1. Mount Drive and load the submitted branch

The repository supplies the fixed split and training code. Drive supplies the teacher-data archive and stores results that must survive runtime deletion.


In [3]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


In [4]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        print("Local repository changes found; skipped the automatic update.")
    else:
        run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")


Mounted at /content/drive
$ git clone --branch task-3-gender-usage-classification --single-branch https://github.com/TrnLin/MLA2.git /content/MLA2
Repository ready: /content/MLA2
Branch: task-3-gender-usage-classification
Commit: 4337c5af6f6f56478c34c7a1365b5740ed65263e


## 2. Copy the teacher data onto the runtime disk

Training reads images from the Colab disk because repeated reads from mounted Drive are much slower. The original folder structure inside the archive is preserved.


In [5]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [
        name for name in names if Path(name).is_absolute() or ".." in Path(name).parts
    ]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/")
        and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")

    current_images = sum(
        path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*")
    )
    needs_extract = current_images != expected_images or not all(
        path.is_file() for path in required_files
    )
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(
    path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*")
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, "
        f"found {actual_images:,}; missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")


Extracting 44,441 teacher images...
Teacher data ready: 44,441 images


## 3. Verify the runtime before training

This performs data, fold, label-map, package-version, model-shape, and GPU checks without taking an optimiser step.


In [6]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (
    DRIVE_TASK_DIR / "baseline",
    DRIVE_TASK_DIR / "logs",
    DRIVE_TASK_DIR / "results",
):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_baseline import check_task3_baseline_setup

gender_check = check_task3_baseline_setup("gender", root=REPO_DIR, device_name="cuda")
usage_check = check_task3_baseline_setup("usage", root=REPO_DIR, device_name="cuda")

print("GPU:", gender_check["environment"]["gpu"])
print("Gender parameters:", f"{gender_check['parameter_count']:,}")
print("Usage parameters: ", f"{usage_check['parameter_count']:,}")
print("All five official folds are ready for both targets.")
print("Persistent registry:", DRIVE_REGISTRY)


GPU: NVIDIA L4
Gender parameters: 390,181
Usage parameters:  391,209
All five official folds are ready for both targets.
Persistent registry: /content/drive/MyDrive/MLA2/task3/results/runs.csv


## 4. Train the five gender folds

Running this cell is the start action. It stays active until folds 0–4 and the pooled out-of-fold result finish. Progress prints after every epoch. Existing Drive results are kept; this call still creates a new run for every fold, including fold 0.


In [7]:
from fashion.train.task3_baseline import run_task3_baseline_cv

gender_result = run_task3_baseline_cv(
    "gender",
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_result


[task3] starting five-fold E1 baseline for target=gender
[task3] preparing target=gender fold=0: train=26,220, validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_baseline_gender_smallcnn_f0_s2753_e46cd00adf0a_20260830T082833Zf8c1e0; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.6290 train_macro_f1=0.3811 validation_loss=0.7514 validation_macro_f1=0.3240
[task3] target=gender fold=0 epoch=2/30 train_loss=0.4688 train_macro_f1=0.5439 validation_loss=1.0253 validation_macro_f1=0.2755
[task3] target=gender fold=0 epoch=3/30 train_loss=0.4126 train_macro_f1=0.6213 validation_loss=0.4812 validation_macro_f1=0.5509
[task3] target=gender fold=0 epoch=4/30 train_loss=0.3812 train_macro_f1=0.6572 validation_loss=0.4095 validation_macro_f1=0.6034
[task3] target=gender fold=0 epoch=5/30 train_loss=0.3564 train_macro_f1=0.6846 validation_loss=0.

{'target': 'gender',
 'fold_run_ids': ['t3_baseline_gender_smallcnn_f0_s2753_e46cd00adf0a_20260830T082833Zf8c1e0',
  't3_baseline_gender_smallcnn_f1_s2753_e46cd00adf0a_20260830T083708Z143950',
  't3_baseline_gender_smallcnn_f2_s2753_e46cd00adf0a_20260830T084548Z5acbf0',
  't3_baseline_gender_smallcnn_f3_s2753_e46cd00adf0a_20260830T085427Z5d34f9',
  't3_baseline_gender_smallcnn_f4_s2753_e46cd00adf0a_20260830T090303Z41a843'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/baseline/gender/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/baseline/gender/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/baseline/gender/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/baseline/gender/aggregate/confusion_matrix.csv',
 'failure_index_path': '/content/drive/MyDrive/MLA2/task3/baseline/gender/aggregate/failure_index.csv',
 'metrics': {'accuracy': 0.8767583071430751,
  'balanced_accuracy': 0.

## 5. Train the five usage folds

Run this foreground cell after the gender run finishes. It trains a separate scratch model for usage on folds 0–4 and then creates its pooled result.


In [8]:
usage_result = run_task3_baseline_cv(
    "usage",
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_result


[task3] starting five-fold E1 baseline for target=usage
[task3] preparing target=usage fold=0: train=26,219, validation=6,553
[task3] fitting fold-training RGB statistics for target=usage fold=0
[task3] RGB statistics ready for target=usage fold=0
[task3] registered t3_baseline_usage_smallcnn_f0_s2753_b0458638128b_20260830T091143Zc4f554; the first optimiser step may now run
[task3] target=usage fold=0 epoch=1/30 train_loss=0.6160 train_macro_f1=0.2262 validation_loss=0.6566 validation_macro_f1=0.2435
[task3] target=usage fold=0 epoch=2/30 train_loss=0.4258 train_macro_f1=0.3111 validation_loss=0.4538 validation_macro_f1=0.2787
[task3] target=usage fold=0 epoch=3/30 train_loss=0.3842 train_macro_f1=0.3309 validation_loss=0.4647 validation_macro_f1=0.3102
[task3] target=usage fold=0 epoch=4/30 train_loss=0.3560 train_macro_f1=0.3424 validation_loss=0.4372 validation_macro_f1=0.3321
[task3] target=usage fold=0 epoch=5/30 train_loss=0.3304 train_macro_f1=0.3514 validation_loss=0.5915 valid

{'target': 'usage',
 'fold_run_ids': ['t3_baseline_usage_smallcnn_f0_s2753_b0458638128b_20260830T091143Zc4f554',
  't3_baseline_usage_smallcnn_f1_s2753_b0458638128b_20260830T092019Z358580',
  't3_baseline_usage_smallcnn_f2_s2753_b0458638128b_20260830T092857Z8d67ab',
  't3_baseline_usage_smallcnn_f3_s2753_b0458638128b_20260830T093738Zdd75d3',
  't3_baseline_usage_smallcnn_f4_s2753_b0458638128b_20260830T094617Z1d2e79'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/baseline/usage/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/baseline/usage/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/baseline/usage/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/baseline/usage/aggregate/confusion_matrix.csv',
 'failure_index_path': '/content/drive/MyDrive/MLA2/task3/baseline/usage/aggregate/failure_index.csv',
 'metrics': {'accuracy': 0.8929879165140974,
  'balanced_accuracy': 0.35709699184

## 6. Show the two pooled baseline results

These values are observations from the fixed baseline. They must be analysed before a child model is proposed.


In [9]:
import pandas as pd

pd.DataFrame(
    [
        {
            "target": result["target"],
            "macro_f1": result["metrics"]["macro_f1"],
            "accuracy": result["metrics"]["accuracy"],
            "balanced_accuracy": result["metrics"]["balanced_accuracy"],
            "metrics_path": result["metrics_path"],
        }
        for result in (gender_result, usage_result)
    ]
)


,target,macro_f1,accuracy,balanced_accuracy,metrics_path
0,gender,0.711753,0.876758,0.693301,/content/drive/MyDrive/MLA2/task3/baseline/gen...
1,usage,0.373756,0.892988,0.357097,/content/drive/MyDrive/MLA2/task3/baseline/usa...
